📋 **Instructions:**

**🎬 Video:** [Project Part 2: Whitepaper](https://youtu.be/2hOXEEzDRoY)

## Assignment Overview

In the first six lessons you learned how to:

1. **Scrape** Reddit data for a university subreddit
2. **Clean and wrangle** raw text data with pandas
3. **Extract locations** using Named Entity Recognition (NER)
4. **Geoparse** place names into real-world coordinates
5. **Run sentiment analysis** using RoBERTa
6. **Map emotions** spatially with Plotly

Now you apply that full pipeline to **your assigned school** and compare the results to the data you created for JMU. Your goal is to produce:

- A **cleaned, sentiment-annotated dataset** of both your assigned school and JMU
  - You should already have a good version of the JMU data based on lessons 4–5. You can go back and keep editing the JMU file if you want to improve accuracy.
- Maps that show *before* and *after* data cleaning
- Two different maps of the same data that each tell a different story
- A **written analysis** of what the maps reveal
- Upon completion the content will populate the **`whitepaper.html`** page of your team website

The format for this assignment is **fill in the blank**. You will go through and see four types of cells:
1. Headers — you do not need to touch these
2. Code cells — some will run automatically, others will require action on your part to make them work
3. Instruction cells — you do not need to touch these. They disappear when you publish the notebook
4. Content cells — these cells say `[ENTER TEXT HERE]`. You will replace that text with your own text.

---

## ⚠️ Before You Begin — Run the Data Pipeline First

Before opening this notebook, you must run **[`project_part_1_data_pipeline.ipynb`](project_part_1_data_pipeline.ipynb)** to completion. That notebook:

1. Loads your team's Google Sheets location review
2. Applies corrections and exports `{SCHOOL}_geoparsed_cleaned.csv`
3. Runs RoBERTa sentiment and exports `{SCHOOL}_geoparsed_cleaned_sentiment.csv`
4. Commits and pushes the data files to GitHub via a pull request

**This whitepaper notebook will not work correctly until those files exist.**

Once the pipeline is complete, set your school name in the cell below and run it — all subsequent cells will use it automatically.


In [1]:
import os
import pandas as pd
import sys, importlib
import plotly.express as px
import plotly.io as pio


SCHOOL = "VCU"  # ← change to your school: GMU, ODU, UNC, UVA, VCU, VirginiaTech, WM

# ── Load sentiment data (produced by project_part_1_data_pipeline.ipynb) ──────────
_sentiment_path = f'../data/{SCHOOL}/{SCHOOL}_geoparsed_cleaned_sentiment.pickle'

if not os.path.exists(_sentiment_path):
    print(f'⛔ File not found: {_sentiment_path}')
    print("   Run project_part_1_data_pipeline.ipynb first, then re-run this cell.")
    df = None
else:
    df = pd.read_pickle(_sentiment_path)
    print(f'✅ {SCHOOL}  |  {len(df):,} rows  |  {df["place"].nunique():,} unique places')
    print(f'   Columns: {list(df.columns)}')

pio.renderers.default = "notebook_connected"

sys.path.insert(0, "..")
import tests.helpers as _helpers_mod

importlib.reload(_helpers_mod)
from tests.helpers import (
    aggregate_places,
    filter_places,
    jenks_size_classify,
    jenks_count_classify,
    jenks_sentiment_classify,
)

print("✅ Map helpers loaded.")

✅ VCU  |  924 rows  |  270 unique places
   Columns: ['type', 'date', 'score', 'sentences', 'place', 'latitude', 'longitude', 'feature_type', 'admin1_name', 'admin2_name', 'country_name', 'school', 'place_type', 'reviewer', 'roberta_compound']
✅ Map helpers loaded.


📋 **Instructions:**

## Custom Template

Your charts will use a custom template. Refer back to [Lesson 3.3](../lesson_3_introduction_pandas/lesson_3_3_plotly_styling.ipynb) for details on how to create a custom template.

**You must run the code below to load the template into memory. You will need to run the template cell every time you make an update.**

In [2]:
import plotly.graph_objects as go
import plotly.io as pio

# ── Define a custom template ──────────────────────────────────────────────
my_template = go.layout.Template()
my_template.layout = go.Layout(
    paper_bgcolor="white",
    plot_bgcolor="#f0f4f8",
    font={
        "family": "Arial",
        "size": 12,
        "color": "#333333",
    },
    colorway=["#450084", "#4B9CD3"], #First color is JMU, second color is your school
    title_font={
        "family": "Times New Roman",
        "size": 16,
        "color": "#222222",
    },
    xaxis={
        "showgrid": True,
        "gridcolor": "#e0e0e0",
        "tickfont": {
            "family": "Arial",
            "size": 10,
            "color": "#555555",
        },
    },
    yaxis={
        "showgrid": True,
        "gridcolor": "#e0e0e0",
        "tickfont": {
            "family": "Arial",
            "size": 10,
            "color": "#555555",
        },
    },
)

# Register the template by name so any px function can use it
pio.templates["my_template"] = my_template

📋 **Instructions:**
The title below is a generic one. Please change it to something catchier. This should be done last.


# Project Part 2: Analysis & Whitepaper [Change Title]

## Introduction

📋 **Instructions:**

**Task:** Write an introduction to the entire whitepaper. 
**Length:** 5-8 sentences
**Detail:**
This should be written **last** when you already know your results. It should include the following:

- A description of both institutions and some of their most common characteristics.
  - For example, you might focus on JMU being a residential school with a vibrant on-campus life while another school might be more known to be a commuter school.
- An overview of the two datasets based on your reading of them. This should give some indication of what this data is about generally.
  - For example, does one school appear to talk about football more versus another one?
- A hypothesis about what different types of locations you expect to find for each school and how you would see this on the map.
- An overview of the actual results making clear whether they affirmed or contradicted the hypothesis.
  - For example, Given the high number of students from Northern Virginia, we hypothesized that there would be a closer connection to NoVa and regional affairs at JMU versus School X which only attracts students from the surrounding area.
- Valuable insight gained from the maps.

> 👉 **Note:** *You are essentially laying out your entire project in one paragraph.*

[This whitepaper examines how the geographic environments of James Madison University (JMU) and Virginia Commonwealth University (VCU) are reflected in Reddit discussions about each university. Although both schools provide similar college experiences, JMU is located in a traditional college town, while VCU is situated in the urban environment of Richmond, Virginia. By analyzing Reddit posts and comments from 2012 to 2026, this study explores the topics people discuss, the locations they mention, and the sentiment associated with those places. The analysis uses keyword trends, timeline activity, geographic mapping, and sentiment scores to identify patterns within each university's online community. We hypothesized that VCU's urban setting would produce a wider range of geographic discussions and more negative sentiment, while JMU discussions would be more positive and concentrated around campus life. The findings reveal similarities and differences between the two universities, including how major events and specific campus locations influence online conversations. This comparison provides insight into how university environments may shape the way students and community members discuss their experiences online.]

## Hypothesis

📋 **Instructions:**

Formulate a working theory (hypothesis) of how the two corpora differ or compare and explain how you might see that reflected in the data. 

**⚠️ Important Requirements:**
- The hypothesis **must** be related to space and sentiment
- Example of what **NOT** to do: "JMU students love math classes more than UNC students" (not spatial and cannot be answered by our data)

#### 💡 Sample Hypotheses

**Regional Scale Examples:**
1. UNC attracts more students from the southeast than JMU and therefore more positive sentiments will appear across South Carolina, Georgia, Alabama, and Florida.

**Local Scale Examples:**

2. JMU has a very positive college atmosphere, and therefore sentiment around campus will appear more positive than that of UNC.

Note that these hypotheses operate at two different spatial scales: regional and local.

[We hypothesize that VCU's urban environment will produce a wider range of geographic discussions and more negative sentiment surrounding locations connected to city life, public safety, and surrounding neighborhoods. In comparison, we expect JMU's more traditional college-town environment to show more positive sentiment concentrated around campus spaces and locations associated with student life. We anticipate that these differences will be reflected in the geographic distribution of Reddit posts and comments, as well as the overall sentiment associated with specific locations at each university.]

## Corpus Description

📋 **Instructions:**

**Task:** Give a global overview of both corpora. 
**Length:** 5-8 sentences
**Detail:** Your overview should include a mix of quantitative and qualitative data. You should include the number of total posts and comments and how many years the thread has been active. 
Also provide more qualitative insights into what kinds of topics people appear to post about. You can do this by filtering the `_raw.csv` data file by posts. The goal is to give the reader a better sense of what the data is like.


[This study compares Reddit discussions about James Madison University (JMU) and Virginia Commonwealth University (VCU) from 2012 to 2026, examining how students and community members discuss their university experiences. The JMU dataset contains 6,948 total records, including 497 posts and 6,451 comments, while the VCU dataset contains 8,509 records, including 493 posts and 8,016 comments. JMU discussions span from February 28, 2012, through April 28, 2026, while VCU discussions span from April 5, 2012, through April 30, 2026. Common discussion topics include academics, campus life, university events, football, parties, tuition, and COVID-19. Although both communities discuss shared college experiences, their conversations also reflect differences in their campus environments and student communities. Comparing these discussions allows us to explore how the geographic settings of JMU and VCU may influence the topics people discuss and the emotions associated with specific locations.]

## Corpus Analysis

### Key Moments


📋 **Instructions:**

**Task:** Provide a data driven analysis of key moments in the Reddit thread
**Length:** 5-8 sentences
**Detail:** The graph below shows a timeline of combined posts and comments for each school. Analyze the peaks for each school and see where posting overlaps and see where it diverges. Institutions might have a shared history around certain topics like Covid, but they may also diverge when it comes to topics like tuition or a big football victory. Your goal is to figure out the data under those "peaks" by going back and reading the raw csv file for those dates to get a better sense of what is going on. Then write up the shared and divergent key moments for each school.

As chart similar to the one below was covered in lesson 3_2. You will have to designate a color for your own school. At the moment it is set to UNC light blue.

In [3]:

# ── Parameters to change ──────────────────────────────────────────────────
INTERVAL     = "W"        # ← change to 'M' (monthly) or 'Y' (yearly)


# ── Load both schools' raw data ───────────────────────────────────────────
_df_jmu    = pd.read_pickle("../data/JMU/JMU_raw_cleaned.pickle")
_df_school = pd.read_pickle(f'../data/{SCHOOL}/{SCHOOL}_raw.pickle')

_df_jmu["school"]    = "JMU"
_df_school["school"] = SCHOOL

_df_combined = pd.concat([_df_jmu, _df_school], ignore_index=True)

# ── Group by interval and school, count all posts + comments ──────────────
_df_combined["interval"] = _df_combined["date"].dt.to_period(INTERVAL).dt.to_timestamp()
_interval_counts = _df_combined.groupby(["interval", "school"], as_index=False).size()

# ── Plot ──────────────────────────────────────────────────────────────────
fig = px.line(
    _interval_counts,
    x="interval",
    y="size",
    color="school",
    title=f'Posts and Comments Over Time: JMU vs {SCHOOL}',
    labels={"interval": "Time Period", "size": "Number of Posts/Comments", "school": "School"},
    markers=True,
    template="my_template",
)
fig.update_layout(
    xaxis_title="Time Period",
    yaxis_title="Number of Posts/Comments",
    legend_title="School",
)
fig.show()


In [15]:
# Find the busiest weeks for each university

for school, path in [
    ("JMU", "../data/JMU/JMU_raw_cleaned.pickle"),
    ("VCU", "../data/VCU/VCU_raw.pickle")
]:
    data = pd.read_pickle(path)
    print("\n", school)
    print(data.columns.tolist())
    display(data.head(3))


 JMU
['type', 'text', 'date', 'score']


,type,text,date,score
0,post,"From a Duck Fan and former OMB member, your ba...",2025-12-22,400
1,comment,MRD are awesome. This was only ½ of the band t...,2025-12-22,87
2,comment,"When I went to JMU, students went to the footb...",2025-12-22,60



 VCU
['type', 'text', 'date', 'score']


,type,text,date,score
0,post,VCU Health suspends gender-affirming care for ...,2025-01-31,1062
1,comment,Please be mindful of the rules of this subredd...,2025-02-02,1
2,comment,I feel bad for VCU and their patients. I feel ...,2025-02-01,11


In [16]:
# Identify the 5 busiest weeks and the discussions from those weeks

for school, path in [
    ("JMU", "../data/JMU/JMU_raw_cleaned.pickle"),
    ("VCU", "../data/VCU/VCU_raw.pickle")
]:
    data = pd.read_pickle(path)
    data["date"] = pd.to_datetime(data["date"])

    weekly = data.set_index("date").resample("W").size()
    busiest_weeks = weekly.nlargest(5)

    print(f"\n========== {school} BUSIEST WEEKS ==========")

    for week, count in busiest_weeks.items():
        print(f"\nWeek ending {week.date()} — {count} discussions")

        posts = data[
            (data["date"] > week - pd.Timedelta(days=7)) &
            (data["date"] <= week)
        ].sort_values("score", ascending=False)

        display(posts[["type", "date", "score", "text"]].head(8))


========== JMU BUSIEST WEEKS ==========

Week ending 2020-09-06 — 491 discussions


,type,date,score,text
102,post,2020-09-01,330,Alger's response to hitting over 500 cases in ...
110,post,2020-09-02,310,Virginia schools be like
267,post,2020-09-06,232,"Leaving Hburg today, and this'll be my last im..."
340,post,2020-09-01,215,Welp
366,post,2020-09-02,208,We did it Patrick! We Re-Opened Campus!
370,post,2020-09-06,206,Well that was fast
379,post,2020-09-04,201,JMU Nazi Zombies Edit
519,post,2020-08-31,184,Cautious optimism.



Week ending 2020-08-30 — 460 discussions


,type,date,score,text
459,post,2020-08-28,198,..........
642,post,2020-08-29,161,Alger be like:
624,post,2020-08-26,159,The home of the Dukes in MS Flight Simulator
749,post,2020-08-29,158,"""ThE nUmBeR oF nEw PoSiTiVe CaSeS iS nOt A sIn..."
824,post,2020-08-28,154,"If you're in an overcrowded classroom, TAKE PH..."
1107,post,2020-08-28,148,Roll Dukes ✊
1022,post,2020-08-27,145,I give it a week
1235,post,2020-08-24,130,"Eagle field last night. No social Distancing, ..."



Week ending 2020-08-23 — 155 discussions


,type,date,score,text
1052,post,2020-08-20,147,@the selfish people who plan to party...
2221,post,2020-08-17,108,Can't wait for Alger and the rest of the admin...
3052,post,2020-08-22,92,[Enraged parents have entered the chat]
3289,post,2020-08-21,86,"""if you choose to host or attend social gather..."
3582,post,2020-08-22,85,The vibe I'm getting from JMU
3699,post,2020-08-20,79,VCU reports 25 confirmed student COVID-19 case...
3290,comment,2020-08-21,76,As they load you into dorms like stuffing saus...
4041,post,2020-08-19,75,Dukes Bar(Habana's) 2 days ago


... (10 outputs left out)


Week ending 2025-09-21 — 169 discussions


,type,date,score,text
1532,post,2025-09-18,161,Hibbs hall is scary
1944,post,2025-09-21,141,Experience walking your animals on/ around cam...
2156,post,2025-09-20,136,"William, you left stuff at Mama J's! Mama J's ..."
2308,post,2025-09-16,129,Some of you need to learn how to park
4099,comment,2025-09-17,114,My theory is that it's due to all the threats ...
2309,comment,2025-09-16,110,That's hardly the worst I've seen this morning
3689,post,2025-09-15,87,"I lost my emotional support shark Hey guys, I ..."
4098,post,2025-09-17,78,Did I miss something? I'm a virtual student bu...



Week ending 2025-02-23 — 167 discussions


,type,date,score,text
2568,post,2025-02-20,117,Saw these snowmen on a bench in Monroe Park :)...
6616,comment,2025-02-20,83,Let's be honest - No one actually has their sh...
7872,comment,2025-02-22,67,The large pedestrian area between shafer and t...
5505,post,2025-02-20,63,fucking finally why did they even wait so long
5744,post,2025-02-19,60,Ranking on campus eating options Valid or nah?
7235,comment,2025-02-19,57,they cancelled last week and my poor professor...
6615,post,2025-02-20,56,Will vcu put student safety first The snow isn...
5746,comment,2025-02-19,51,I don't know if he's still there this semester...


In [17]:
# Verify corpus totals and date ranges

for school, path in [
    ("JMU", "../data/JMU/JMU_raw_cleaned.pickle"),
    ("VCU", "../data/VCU/VCU_raw.pickle")
]:
    data = pd.read_pickle(path)
    data["date"] = pd.to_datetime(data["date"])

    print(f"\n--- {school} ---")
    print("Total records:", len(data))
    print("Posts:", (data["type"] == "post").sum())
    print("Comments:", (data["type"] == "comment").sum())
    print("First date:", data["date"].min().date())
    print("Last date:", data["date"].max().date())


--- JMU ---
Total records: 6948
Posts: 497
Comments: 6451
First date: 2012-02-28
Last date: 2026-04-28

--- VCU ---
Total records: 8509
Posts: 493
Comments: 8016
First date: 2012-04-05
Last date: 2026-04-30


[The timeline reveals that major events and university-specific issues influenced Reddit activity at both JMU and VCU. JMU experienced its largest spike during the week ending September 6, 2020, with 491 posts and comments. Many discussions during this period focused on the university's response to rising COVID-19 cases, campus reopening, crowded classrooms, and concerns about students following social distancing guidelines. VCU's largest spike occurred during the week ending February 2, 2025, with 217 posts and comments, largely surrounding reports that VCU Health was suspending gender-affirming care and discussions about the university's response. VCU also experienced increased activity in May 2024, when Reddit discussions focused on campus protests, police responses, and calls for the university president's resignation. Both universities experienced another noticeable increase in activity during August 2020, when conversations centered on campus reopening, COVID-19 cases, and student behavior. These patterns demonstrate that Reddit activity often increased during periods of controversy, uncertainty, and major changes affecting each university community.]

### Key Topics

📋 **Instructions:**

**Task:** Provide a data-driven analysis of the most discussed topics in each subreddit.
**Length:** 5–8 sentences
**Detail:** The chart below shows how frequently each keyword appears across posts and comments for both schools. Your first and most important task is to **decide which keywords are worth comparing**. A keyword like `class` will appear in almost every academic subreddit and tell you nothing interesting; a keyword like `tailgate` or `dining hall` may reveal something specific about campus culture at one school.

To find good keywords:
- Skim 20–30 posts from each school's `_raw.csv` file and note recurring topics, complaints, or celebrations
- Think about what is *unique* to each school's context (athletics, geography, campus life, local events)
- Avoid keywords so generic they appear everywhere (`good`, `people`, `year`) or so rare they barely register

Once you have a chart you find meaningful, analyze it: which topics are **shared** across both schools, and which are **distinctive** to one? Do the differences reflect something meaningful about each institution?

> 👉 **Note:** *Because the two corpora are different sizes, the chart uses counts per 1,000 posts so the schools are directly comparable.*


In [4]:

# ── Keywords to compare — change this list ───────────────────────────────
# Tip: skim each school's _raw.csv first to find topics that are actually
# discussed. Generic words ('good', 'people') or rare words produce flat bars.
KEYWORDS = ["tuition", "covid", "party", "football", "basketball", "class", "zoom", "campus"]

# ── Load raw data for both schools ────────────────────────────────────────
_df_jmu    = pd.read_pickle("../data/JMU/JMU_raw_cleaned.pickle")
_df_school = pd.read_pickle(f'../data/{SCHOOL}/{SCHOOL}_raw.pickle')

# ── Count keyword mentions and normalize per 1,000 posts ─────────────────
# Normalizing makes the two corpora directly comparable even though they
# have different numbers of posts and comments.
_jmu_total    = len(_df_jmu)
_school_total = len(_df_school)

_rows = []
for kw in KEYWORDS:
    _jmu_hits    = _df_jmu["text"].str.contains(kw, case=False, na=False).sum()
    _school_hits = _df_school["text"].str.contains(kw, case=False, na=False).sum()
    _rows.append({"keyword": kw, "school": "JMU",    "per_1000": _jmu_hits    / _jmu_total    * 1000})
    _rows.append({"keyword": kw, "school": SCHOOL,   "per_1000": _school_hits / _school_total * 1000})

_keyword_df = pd.DataFrame(_rows)

# ── Plot ──────────────────────────────────────────────────────────────────
fig = px.bar(
    _keyword_df,
    x="keyword",
    y="per_1000",
    color="school",
    barmode="group",
    title=f'Keyword Frequency per 1,000 Posts: JMU vs {SCHOOL}',
    labels={"per_1000": "Mentions per 1,000 Posts/Comments",
            "keyword": "Keyword", "school": "School"},
    template="my_template"
)
fig.update_layout(
    xaxis_title="Keyword",
    yaxis_title="Mentions per 1,000 Posts/Comments",
    legend_title="School",
    xaxis={"categoryorder": "total descending"},
)
fig.show()


[The keyword frequency chart reveals several differences in the topics discussed within the JMU and VCU Reddit communities. Both universities frequently discuss classes and campus life, showing that academics and the college experience are important topics across both communities. However, VCU has a higher frequency of class-related mentions, while JMU has more mentions of campus, COVID, parties, and football. JMU's stronger emphasis on campus life, parties, and football may reflect its traditional college-town atmosphere and the importance of shared student experiences. In comparison, VCU's higher frequency of class-related discussions may suggest a stronger focus on academic experiences within its urban university environment. Tuition and Zoom appear at relatively similar levels for both schools, suggesting some shared concerns and experiences. Overall, the chart suggests that JMU discussions are more centered on traditional campus culture, while VCU conversations place greater emphasis on academics.]

### Keyword Trends


📋 **Instructions:**

**Task:** Identify up to three keywords that show the most meaningful difference — or the most surprising similarity — between the two schools and plot how their usage evolved over time.
**Length:** 5–8 sentences
**Detail:** Looking at your Key Topics bar chart, identify up to three keywords that have the highest analytical payoff: either a keyword one school uses far more than the other, or one that both schools spike around the same time (suggesting a shared event like COVID or a regional news story). You may use fewer than three if two keywords already tell a complete story — more lines can make the chart harder to read.

To decide which keywords to keep:
- A keyword is worth including if it produces at least one noticeable peak or divergence you can explain
- If two keywords behave nearly identically, drop one — they are telling the same story
- A flat line for both schools is not interesting unless the *absence* of a topic is itself meaningful

**Adjusting the chart:** Two variables control how the chart looks:
- `TREND_INTERVAL` — the time grouping. Finer intervals (`'W'` weekly, `'M'` monthly) show more detail but can produce spiky lines when post volume is low. Coarser intervals (`'Q'` quarterly, `'Y'` yearly) smooth the lines and reduce distortion because each bucket has more posts to normalize against. Start with `'Q'` and zoom in only if a quarterly chart already shows a clear story.
- `MIN_PERIOD_POSTS` — any interval with fewer total posts than this threshold is dropped before normalization. Raise it to remove more noisy periods; lower it to keep them.

**Color** distinguishes the two schools; **line pattern** (solid vs. dashed vs. dotted) distinguishes the keywords. Your written analysis should name each peak, explain what was happening at that time, and say what it reveals about each institution.

**Annotations** help point out important patterns in the data. Two have been created for you. Adjust them as necessary. You may also opt to delete them. See [Lesson 3.3](../lesson_3_introduction_pandas/lesson_3_3_plotly_styling.ipynb) for details.





In [5]:
# ── Pick 1–3 keywords from your Key Topics chart ─────────────────────────
# Choose keywords that produced the most interesting peaks or divergences.
# You can use fewer than 3 if two keywords already tell a complete story.
TREND_KEYWORDS = ["covid", "football", "tuition"]  # ← change these (1–3 keywords)

# ── Time interval for this chart ──────────────────────────────────────────
# Coarser intervals (Q, Y) pool more posts per bucket, which reduces
# normalization distortion and produces smoother lines.
# Options: 'W' (weekly), 'M' (monthly), 'Q' (quarterly), 'Y' (yearly)
TREND_INTERVAL = "Y"  # ← change if needed (default: quarterly)

# ── Minimum posts per interval to include in the chart ───────────────────
# Intervals below this threshold are dropped before normalizing.
# Raise to suppress noisy periods; lower to include them.
MIN_PERIOD_POSTS = 10  # ← change if needed

# ── Load raw data for both schools ────────────────────────────────────────
_df_jmu    = pd.read_pickle("../data/JMU/JMU_raw_cleaned.pickle")
_df_school = pd.read_pickle(f'../data/{SCHOOL}/{SCHOOL}_raw.pickle')

_df_jmu["school"]    = "JMU"
_df_school["school"] = SCHOOL

_df_combined = pd.concat([_df_jmu, _df_school], ignore_index=True)

_df_combined["interval"] = _df_combined["date"].dt.to_period(TREND_INTERVAL).dt.to_timestamp()

# ── Normalize per 1,000 — drop intervals with too few posts ──────────────
# A month with 1 total post and 1 covid hit = 1,000 per 1,000: not meaningful.
_period_totals = (_df_combined.groupby(["interval", "school"])
                               .size()
                               .reset_index(name="_total"))
_period_totals = _period_totals[_period_totals["_total"] >= MIN_PERIOD_POSTS]

_rows = []
for kw in TREND_KEYWORDS:
    _mask = _df_combined["text"].str.contains(kw, case=False, na=False)
    _hits = (_df_combined[_mask]
             .groupby(["interval", "school"])
             .size()
             .reset_index(name="_hits"))
    _m = _hits.merge(_period_totals, on=["interval", "school"])
    _m["per_1000"] = _m["_hits"] / _m["_total"] * 1000
    _m["keyword"] = kw
    _rows.append(_m[["interval", "school", "keyword", "per_1000"]])

_trend_df = pd.concat(_rows, ignore_index=True)

# ── Plot — color = school, line pattern = keyword ─────────────────────────
fig = px.line(
    _trend_df,
    x="interval",
    y="per_1000",
    color="school",
    line_dash="keyword",
    title=f'Keyword Trends Over Time: JMU vs {SCHOOL}',
    labels={"interval": "Time Period", "per_1000": "Mentions per 1,000 Posts/Comments",
            "school": "School", "keyword": "Keyword"},
    markers=True,
    template="my_template"
)
fig.update_layout(
    xaxis_title="Time Period",
    yaxis_title="Mentions per 1,000 Posts/Comments",
)

fig.add_vline(
    x="2020-08-01",
    line_dash="dash",
    line_color="#450084",
    line_width=1,
)


fig.add_annotation(
    x="2020-08-01",
    y=25,
    text="RANDOM ANNOTATION 1",
    showarrow=True,
    arrowhead=2,
    ax=-80,
    ay=-60,
)

fig.add_vline(
    x="2021-08-01",
    line_dash="dash",
    line_color="#4B9CD3",
    line_width=1,
)

fig.add_annotation(
    x="2021-08-01",
    y=25,
    text="RANDOM ANNOTATION 2",
    showarrow=True,
    arrowhead=2,
    ax=80,
    ay=-60,
)

fig.show()

[The keyword trends reveal how conversations about COVID, football, and tuition changed over time within the JMU and VCU Reddit communities. COVID-related discussions increased significantly at both universities around 2021, with JMU reaching approximately 54 mentions per 1,000 posts and comments compared with VCU's peak of approximately 26. This shared increase suggests that the pandemic was an important topic across both university communities, although the intensity of discussion differed. Following 2021, COVID mentions declined at both schools, reflecting a decrease in the prominence of pandemic-related conversations over time. Football discussions show a different pattern, with JMU experiencing noticeable increases in 2015 and again in 2025 and 2026, while VCU's trend does not show the same pattern of large increases. Tuition mentions remain comparatively low and fluctuate throughout the timeline. Overall, these trends demonstrate how some topics, such as COVID, affected both communities, while others, such as football, appeared more prominent in one university's discussions.]

## Spatial Data Cleaning

### JMU

📋 **Instructions:**

**Task:** Give a brief description of the process of cleaning locations for JMU. You may refer back to your experience in previous lessons. 
**Length:** 5-8 Sentences
**Description:** You want to demonstrate that you read through the data by providing details on where the geoparser made important mistakes and explain why it made those errant inferences. For example, in the demonstration data the geoparser confused JMU with Jiamusi Songjiang International Airport in China because it has the airport code JMU. This is a big mistake because the data is about JMU the university not the airport. 
Finally, were there any cases where you had a hard time determining whether something should be a location or not. For example, there were a number of references to NJ (New Jersey) drivers in the JMU data, did you list New Jersey as a location or did you remove these because they were references to people from New Jersey?

[The initial JMU location data contained 322 unique places before cleaning. After reviewing and cleaning the geographic information, the dataset was reduced to 282 unique places, removing 40 entries. One noticeable issue was that some locations were incorrectly mapped to Madagascar, even though the conversations were related to JMU and its surrounding communities. These locations demonstrated how the geocoding process could assign incorrect geographic coordinates to place references that did not actually belong in the study area. The before-and-after maps show how cleaning helped refine the geographic distribution and made Harrisonburg a clearer concentration in the data. Other regional locations, including Richmond, Charlottesville, Washington, Baltimore and Philadelphia, remained represented in the cleaned dataset. Overall, reviewing and correcting these geographic errors was important for making the maps more accurate and ensuring the analysis reflected locations genuinely connected to JMU.]

📋 **Instructions:**
The cell below creates two maps: JMU - Data *Before* and JMU Data *After*. Modify the styling parameters to maximize the difference between the two maps. If you have not completed the cleaned JMU map from Lesson 4 this will fallback to a backup. You will not receive credit for the backup file.

In [6]:
# ── Parameters you can change ─────────────────────────────────────────────
MAP_STYLE  = "carto-positron"    # ← try 'carto-darkmatter', 'open-street-map'
COLOR      = "#4c78a8"           # ← try any hex color or CSS name, e.g. 'steelblue'
CENTER_LAT = 38.44               # ← map center latitude
CENTER_LON = -78.87              # ← map center longitude
ZOOM       = 6                   # ← zoom level
SIZE_MAX   = 18                  # ← maximum bubble size

# ── Load data ──────────────────────────────────────────────────────────────
_before_raw  = pd.read_csv("../data/JMU/JMU_geoparsed_long.csv")
_after_path  = "../data/JMU/JMU_geoparsed_cleaned.pickle"
_backup_path = "../data/JMU/JMU_geoparsed_cleaned_backup.pickle"

if os.path.exists(_after_path):
    _after_raw, _after_label = pd.read_pickle(_after_path), "After cleaning"
else:
    _after_raw   = pd.read_pickle(_backup_path)
    _after_label = "After cleaning (⚠️ backup — complete Part 3 to replace this)"
    print("⚠️  JMU_geoparsed_long_cleaned.csv not found — using backup.")

df_before = aggregate_places(_before_raw)
df_after  = aggregate_places(_after_raw)
print(f'Before: {len(df_before):,} unique places  |  '
      f'After: {len(df_after):,}  |  Removed: {len(df_before) - len(df_after):,}')

# ── Shared map options ─────────────────────────────────────────────────────
_base = {
    "lat": "latitude",
    "lon": "longitude",
    "size": "size_class",
    "hover_name": "place",
    "hover_data": {
        "location_count": True,
        "place_type": True,
        "size_class": False,
        "latitude": False,
        "longitude": False,
    },
    "color_discrete_sequence": [COLOR],
    "size_max": SIZE_MAX,
    "map_style": MAP_STYLE,
    "center": {
        "lat": CENTER_LAT,
        "lon": CENTER_LON,
    },
    "zoom": ZOOM,
    "height": 460,
}

# ── Before map ─────────────────────────────────────────────────────────────
jenks_size_classify(df_before, 4)
px.scatter_map(
    df_before,
    title="JMU — Before cleaning",  # ← change this title
    **_base,
).update_layout(
    margin={
        "r": 0,
        "t": 50,
        "l": 0,
        "b": 0,
    }
).show()

# ── After map ──────────────────────────────────────────────────────────────
jenks_size_classify(df_after, 4)
px.scatter_map(
    df_after,
    title=f'JMU — {_after_label}',  # ← change this title
    **_base,
).update_layout(
    margin={
        "r": 0,
        "t": 50,
        "l": 0,
        "b": 0,
    }
).show()

Before: 322 unique places  |  After: 282  |  Removed: 40


### VCU <- Change this


📋 **Instructions:**

**Task:** Give a brief description of the process of cleaning locations for your school. This mirrors the description you already gave for JMU.
**Length:** 5-8 Sentences
**Description:** You want to demonstrate that you read through the data by providing details on where the geoparser made important mistakes and explain why it made those errant inferences. You also want to discuss edge cases where you had a hard time determining whether something was a location or not. Finally, consider hyperlocal places. What were some locations that students spoke about that required some research to figure out. For example, most people outside of JMU don't know what D-Hall is. What is the equivalent of that for your school?

[The initial VCU location data contained 303 unique places before cleaning. After reviewing and correcting the geographic information, the dataset was reduced to 270 unique places, a decrease of 33. One noticeable issue was that some locations were incorrectly geocoded to Minnesota, even though the conversations focused on VCU and its surrounding communities in Richmond, Virginia. This demonstrates how the geocoding process can assign incorrect coordinates to place references when their context is not properly considered. Reviewing these errors helped make the geographic data more accurate and relevant to the university. The cleaned dataset also retained recognizable, hyperlocal locations such as Monroe Park and Gladding Residence Center, which are closely connected to VCU’s campus environment and student experiences. Keeping these specific locations in the analysis helps capture conversations about the places students actually know and interact with, rather than relying only on broader city-level locations.]

📋 **Instructions:**
The cell below sets your school variable and verifies that the required data file exists. Make sure you change `UNC` to your school, otherwise the visualizations will not work.

> 👉 **Note:** *Before running this cell, make sure you have completed [project_part_1_data_pipeline.ipynb](project_part_1_data_pipeline.ipynb) for your school. That notebook cleans your data and runs sentiment analysis — without it, the data file this cell checks for will not exist.*



In [7]:
import os


_sentiment_path = f'../data/{SCHOOL}/{SCHOOL}_geoparsed_cleaned_sentiment.pickle'

if not os.path.exists(_sentiment_path):
    print(f'⛔ File not found: {_sentiment_path}')
    print("   Run project_part_1_data_pipeline.ipynb first, then re-run this cell.")
else:
    print(f'✅ {SCHOOL} data found — ready to map.')


✅ VCU data found — ready to map.


📋 **Instructions:**
The cell below creates two maps: Your school data *Before* and Your school data *After*. Modify the styling parameters to maximize the difference between the two maps. 

In [8]:
# ── Parameters you can change ─────────────────────────────────────────────
MAP_STYLE  = "carto-positron"    # ← try 'carto-darkmatter', 'open-street-map'
COLOR      = "#4c78a8"           # ← try any hex color or CSS name, e.g. 'steelblue'
CENTER_LAT = 38.44               # ← map center latitude
CENTER_LON = -78.87              # ← map center longitude
ZOOM       = 4                   # ← zoom level
SIZE_MAX   = 18                  # ← maximum bubble size

# ── Load data ──────────────────────────────────────────────────────────────
_before_raw  = pd.read_csv(f'../data/{SCHOOL}/{SCHOOL}_geoparsed_long.csv')
_after_path  = f'../data/{SCHOOL}/{SCHOOL}_geoparsed_cleaned.pickle'
_backup = "../data/UNC/UNC_geoparsed_cleaned.pickle"

if os.path.exists(_after_path):
    _after_raw, _after_label = pd.read_pickle(_after_path), "After cleaning"
elif os.path.exists(_backup):
    _after_raw, _after_label = pd.read_pickle(_backup), "After cleaning backup"
    print(
        f"⚠️ Data for {SCHOOL} not found. Using UNC backup sentiment — complete the data pipeline to replace this."
    )
else:
    raise FileNotFoundError(
        f'No cleaned data found for {SCHOOL}. Run project_part_1_data_pipeline.ipynb first.'
    )





df_before = aggregate_places(_before_raw)
df_after  = aggregate_places(_after_raw)
print(f'Before: {len(df_before):,} unique places  |  '
      f'After: {len(df_after):,}  |  Removed: {len(df_before) - len(df_after):,}')

# ── Shared map options ─────────────────────────────────────────────────────
_base = {
    "lat": "latitude",
    "lon": "longitude",
    "size": "size_class",
    "hover_name": "place",
    "hover_data": {
        "location_count": True,
        "place_type": True,
        "size_class": False,
        "latitude": False,
        "longitude": False,
    },
    "color_discrete_sequence": [COLOR],
    "size_max": SIZE_MAX,
    "map_style": MAP_STYLE,
    "center": {
        "lat": CENTER_LAT,
        "lon": CENTER_LON,
    },
    "zoom": ZOOM,
    "height": 460,
}

# ── Before map ─────────────────────────────────────────────────────────────
jenks_size_classify(df_before, 4)
px.scatter_map(
    df_before,
    title=f'{SCHOOL} — Before cleaning',  # ← change this title
    **_base,
).update_layout(
    margin={
        "r": 0,
        "t": 50,
        "l": 0,
        "b": 0,
    }
).show()

# ── After map ──────────────────────────────────────────────────────────────
jenks_size_classify(df_after, 4)
px.scatter_map(
    df_after,
    title=f'{SCHOOL} — {_after_label}',  # ← change this title
    **_base,
).update_layout(
    margin={
        "r": 0,
        "t": 50,
        "l": 0,
        "b": 0,
    }
).show()

Before: 303 unique places  |  After: 270  |  Removed: 33


## Sentiment Overview

### JMU

📋 **Instructions:**

**Task**: Create an "overview" map of sentiment for JMU that highlights the kinds of places people talk about and the strongest emotions.
**Length**: 4-5 sentences
**Description**: Before going into comparative analysis with your school, you want to create an overview of both schools. You should create a map that provides the "best" overview. This can be local, regional, national, or global. It all depends on what scale the data ceases to be very useful. You may find that some schools have a far more global perspective while others are more regional. You want to provide as much of a general glance in the map without adding extraneous data. Your description is meant to guide your viewer to salient points.

As a practical matter, you are going to create your map first by jumping to the code cell below and then do your write up. Since this will be an essay, it makes more sense to have the description precede the map that is why you are doing things out of order.


[The JMU sentiment map highlights conversations centered around Harrisonburg and locations connected to the university, while also showing references to places outside the immediate campus area. Harrisonburg appears frequently in the dataset, with 44 mentions, demonstrating the importance of the surrounding college-town community in Reddit discussions. Several recognizable campus locations also stand out, including D-Hall, Southview Apartments, and Eagle Hall. Sentiment varies across these locations, with some student housing areas showing more negative scores and university-related locations such as Bridgeforth Stadium showing more positive sentiment. Overall, the map illustrates how Reddit conversations reflect a combination of campus experiences, student housing, university activities, and the surrounding Harrisonburg community.]

In [9]:
# ── Your design decisions — change each value and re-run ─────────────────────
MIN_COUNT       =  2     # ← Decision 1: minimum post count
PLACE_TYPES     =  None  # ← Decision 2: list of place types (e.g. ['City', 'Building']), or None for all
N_SIZE_CLASSES  =  5     # ← Decision 3: number of Jenks size classes
N_COLOR_BUCKETS =  5     # ← Decision 4: number of Jenks color buckets
COLOR_SCALE     =  "RdYlGn"          # ← Decision 5: Plotly color scale name (e.g. 'RdYlGn')
MAP_STYLE       =  "carto-positron"  # ← Decision 6: base map style (e.g. 'carto-positron')
CENTER          =  {  # ← Decision 7: map center
    "lat": 37,
    "lon": -78,
}
ZOOM            =  5     # ← Decision 8: zoom level
SIZE_MAX        =  18    # ← Decision 9: maximum bubble size

# ── Load JMU sentiment data ────────────────────────────────────────────────
_jmu_primary = "../data/JMU/JMU_geoparsed_cleaned_sentiment.pickle"
_jmu_backup  = "../data/JMU/JMU_geoparsed_cleaned_sentiment_backup.pickle"

if os.path.exists(_jmu_primary):
    _df_jmu_raw = pd.read_pickle(_jmu_primary)
elif os.path.exists(_jmu_backup):
    _df_jmu_raw = pd.read_pickle(_jmu_backup)
    print("⚠️  Using JMU backup sentiment — complete the data pipeline to replace this.")
else:
    raise FileNotFoundError("No JMU sentiment data found. Run generate_jmu_backup_sentiment.ipynb.")

# ── Prepare and plot ───────────────────────────────────────────────────────
df_work = filter_places(aggregate_places(_df_jmu_raw, sentiment=True), MIN_COUNT, PLACE_TYPES)
_labels, _cmap = jenks_sentiment_classify(df_work, COLOR_SCALE, N_SIZE_CLASSES, N_COLOR_BUCKETS)
px.scatter_map(
    df_work,
    lat="latitude", lon="longitude",
    size="size_class", color="color_class",
    hover_name="place",
    hover_data={
        "avg_roberta_compound": ":.3f",
        "location_count": True,
        "size_class": False,
        "color_class": False,
        "latitude": False,
        "longitude": False,
    },
    color_discrete_map=_cmap,
    category_orders={
        "color_class": _labels,
    },
    size_max=SIZE_MAX, map_style=MAP_STYLE, center=CENTER, zoom=ZOOM, height=650,
    title="JMU — Sentiment Map",  # ← change this title
).update_layout(
    margin={
        "r": 0,
        "t": 50,
        "l": 0,
        "b": 0,
    }
).show()

#### Significant Locations

📋 **Instructions:**

**Task**: Write an overview of *three* significant locations in the corpus based on your cleaning and analysis
**Length**: 4-5 sentences
**Description**: The description that precedes the map should provide readers with a general guide for what they are looking at. Your description after they see the map should provide an overview of *three* significant locations in the corpus that might be unknown to someone unfamiliar with the school. For example, on the JMU map D-Hall comes out at a significant place. You want to mention what this place is, where it is, and why it elicits so much conversation. It will help steer them towards the narrative you are constructing.


[Three significant locations in the JMU dataset are Southview Apartments, Eagle Hall, and D-Hall. Southview Apartments appears 11 times and has an average sentiment score of -0.347, making it one of the more negatively discussed locations in the dataset. Eagle Hall, a JMU residence hall, appears eight times with an average sentiment score of -0.276. D-Hall, JMU's main dining hall, appears 19 times and has an average sentiment score of -0.150. These locations demonstrate how Reddit conversations reflect students' experiences with campus dining and residential life, providing insight into everyday aspects of the JMU community.]

### VCU <- Change this

📋 **Instructions:**

**Task**: Create an "overview" map of sentiment for your school that highlights the kinds of places people talk about and the strongest emotions.
**Length**: 4-5 sentences
**Description**: Before going into comparative analysis with your school, you want to create an overview of both schools. You should create a map that provides the "best" overview. This can be local, regional, national, or global. It all depends on what scale the data ceases to be very useful. You may find that some schools have a far more global perspective while others are more regional. You want to provide as much of a general glance in the map without adding extraneous data. Your description is meant to guide your viewer to salient points.

As a practical matter, you are going to create your map first by jumping to the code cell below and then do your write up. Since this will be an essay, it makes more sense to have the description precede the map that is why you are doing things out of order.

[The VCU sentiment map shows that conversations are concentrated around Richmond, Virginia, while also extending to other locations throughout the United States and beyond. Richmond is the most frequently mentioned location, reflecting the importance of the city's urban environment in discussions about VCU. Several recognizable campus locations also appear, including Monroe Park and Gladding Residence Center, showing that students discuss specific places connected to their daily experiences. The sentiment scores across these locations vary, but the most frequently mentioned local places generally have scores closer to neutral than strongly positive or negative. Overall, the map highlights how VCU conversations connect both the university's campus environment and the larger city of Richmond.]

In [10]:
# ── Your design decisions — change each value and re-run ─────────────────────
MIN_COUNT       =  2     # ← Decision 1: minimum post count
PLACE_TYPES     =  None  # ← Decision 2: list of place types (e.g. ['City', 'Building']), or None for all
N_SIZE_CLASSES  =  5     # ← Decision 3: number of Jenks size classes
N_COLOR_BUCKETS =  5     # ← Decision 4: number of Jenks color buckets
COLOR_SCALE     =  "RdYlGn"          # ← Decision 5: Plotly color scale name (e.g. 'RdYlGn')
MAP_STYLE       =  "carto-positron"  # ← Decision 6: base map style (e.g. 'carto-positron')
CENTER          =  {  # ← Decision 7: map center
    "lat": 37,
    "lon": -78,
}
ZOOM            =  5     # ← Decision 8: zoom level
SIZE_MAX        =  18    # ← Decision 9: maximum bubble size

# ── Load school sentiment data ─────────────────────────────────────────────
_primary = f"../data/{SCHOOL}/{SCHOOL}_geoparsed_cleaned_sentiment.pickle"
_backup  = "../data/UNC/UNC_geoparsed_cleaned_sentiment.pickle"

if os.path.exists(_primary):
    _df_school_raw = pd.read_pickle(_primary)
    chart_title_prefix = SCHOOL  # Use the original school name
elif os.path.exists(_backup):
    _df_school_raw = pd.read_pickle(_backup)
    chart_title_prefix = "UNC BACKUP"  # Dynamically switch the prefix
    print(f'⚠️ Data for {SCHOOL} not found. Using UNC backup sentiment — complete the data pipeline to replace this.')
else:
    raise FileNotFoundError(
        f"No sentiment data found for {SCHOOL}.  Run project_part_1_data_pipeline.ipynb first."
    )

# ── Prepare and plot ───────────────────────────────────────────────────────
df_work = filter_places(aggregate_places(_df_school_raw, sentiment=True), MIN_COUNT, PLACE_TYPES)
_labels, _cmap = jenks_sentiment_classify(df_work, COLOR_SCALE, N_SIZE_CLASSES, N_COLOR_BUCKETS)
px.scatter_map(
    df_work,
    lat="latitude", lon="longitude",
    size="size_class", color="color_class",
    hover_name="place",
    hover_data={
        "avg_roberta_compound": ":.3f",
        "location_count": True,
        "size_class": False,
        "color_class": False,
        "latitude": False,
        "longitude": False,
    },
    color_discrete_map=_cmap,
    category_orders={
        "color_class": _labels,
    },
    size_max=SIZE_MAX, map_style=MAP_STYLE, center=CENTER, zoom=ZOOM, height=650,
    title=f'{chart_title_prefix} — Sentiment Map',  
).update_layout(
    margin={
        "r": 0,
        "t": 50,
        "l": 0,
        "b": 0,
    }
).show()

In [11]:
# Top 15 VCU locations by number of mentions
vcu_locations = (
    aggregate_places(_df_school_raw, sentiment=True)
    .sort_values("location_count", ascending=False)
)

display(
    vcu_locations[
        ["place", "location_count", "avg_roberta_compound"]
    ].head(15)
)

,place,location_count,avg_roberta_compound
0,Richmond,189,-0.075158
2,Virginia,56,-0.092928
3,United States,45,-0.354299
14,Monroe Park,45,0.068728
6,State of Israel,30,-0.497585
1,"Richmond, Virginia",26,-0.054759
5,Palestine,25,-0.344165
103,"Gladding Residence Center, VCU",17,0.032830
42,"Richmond, VA",15,-0.170243
59,Virginia Commonwealth University,12,-0.108502


#### Significant Locations

📋 **Instructions:**

**Task**: Write an overview of *three* significant locations in the corpus based on your cleaning and analysis
**Length**: 4-5 sentences
**Description**: The description that precedes the map should provide readers with a general guide for what they are looking at. Your description after they see the map should provide an overview of three significant locations in the corpus that might be unknown to someone unfamiliar with the school. For example, on the JMU map D-Hall comes out at a significant place. You want to mention what this place is, where it is, and why it elicits so much conversation. It will help steer them towards the narrative you are constructing.

[Three significant locations in the VCU dataset are Richmond, Monroe Park, and Gladding Residence Center. Richmond leads the dataset with 189 mentions and an average sentiment score of -0.075, demonstrating how frequently conversations extend beyond the university itself into the surrounding city. Monroe Park, located directly across from VCU's Monroe Park Campus, appears 45 times and has a slightly positive average sentiment score of 0.069. Gladding Residence Center, a VCU student residence hall, appears 17 times with an average sentiment score of 0.033. These locations demonstrate how Reddit conversations reflect both the broader Richmond community and specific places that are part of students' everyday campus experiences.]

## Sentiment Analysis

### JMU

📋 **Instructions:**

**Task**: Write an analysis of the sentiment map of JMU that tries to answer your hypothesis.
**Length**: 4-5 sentences
**Description**: After giving your overview and your highlight of three locations, you are going to analyze the sentiment map in terms of your hypothesis. The key is to make sure that the map and your analysis match. Rather than trying to tell a "global" story, you want to tell a detailed story about one specific issue. You want the map to show the minimum number of locations to tell this story. This story will be the evidence for or against your hypothesis. If the evidence contradicts your hypothesis that is a sign of good scholarship, don't be afraid to contradict your earlier assessment.


[The JMU sentiment results show a mix of positive and negative experiences across campus and surrounding locations. Southview Apartments had the most negative average sentiment score at -0.347, followed by Eagle Hall at -0.276 and Harrisonburg at -0.212. These results suggest that some conversations connected to student housing and the surrounding college-town environment were more negative. In contrast, Bridgeforth Stadium had the highest positive sentiment score at 0.406, suggesting a more positive association with JMU athletics and university events. Overall, the findings partially support our hypothesis that JMU discussions would reflect positive campus experiences, but they also reveal negative sentiment connected to specific student living environments.]

In [12]:
# ── Your design decisions — change each value and re-run ─────────────────────
MIN_COUNT        =  2          # ← Decision 1: minimum post count
PLACE_TYPES      =  None       # ← Decision 2: list of place types, or None for all
N_SIZE_CLASSES   =  5          # ← Decision 3: number of Jenks size buckets (count)
N_COLOR_BUCKETS  =  5          # ← Decision 4: number of Jenks color buckets (sentiment)
COLOR_SCALE      =  "RdYlGn"   # ← Decision 5: color scale shared by both maps
MAP_STYLE        =  "carto-positron"  # ← Decision 6: base map style
CENTER           =  {  # ← Decision 7: map center
    "lat": 37,
    "lon": -78,
}
ZOOM             =  5          # ← Decision 8: zoom level
SIZE_MAX         =  18         # ← Decision 9: maximum bubble size

# ── Load JMU sentiment data ────────────────────────────────────────────────
_jmu_primary = "../data/JMU/JMU_geoparsed_cleaned_sentiment.pickle"
_jmu_backup = "../data/JMU/JMU_geoparsed_cleaned_sentiment_backup.pickle"

if os.path.exists(_jmu_primary):
    _df_jmu_raw = pd.read_pickle(_jmu_primary)
elif os.path.exists(_jmu_backup):
    _df_jmu_raw = pd.read_pickle(_jmu_backup)
    print(
        "⚠️  Using JMU backup sentiment — complete the data pipeline to replace this."
    )
else:
    raise FileNotFoundError(
        "No JMU sentiment data found. Run generate_jmu_backup_sentiment.ipynb."
    )

# ── Load school sentiment data ─────────────────────────────────────────────
_primary = f"../data/{SCHOOL}/{SCHOOL}_geoparsed_cleaned_sentiment.pickle"
_backup = "../data/UNC/UNC_geoparsed_cleaned_sentiment.pickle"

if os.path.exists(_primary):
    _df_school_raw = pd.read_pickle(_primary)
    chart_title_prefix = SCHOOL  # Use the original school name
elif os.path.exists(_backup):
    _df_school_raw = pd.read_pickle(_backup)
    chart_title_prefix = "UNC BACKUP"  # Dynamically switch the prefix
    print(
        f"⚠️ Data for {SCHOOL} not found. Using UNC backup sentiment — complete the data pipeline to replace this."
    )
else:
    raise FileNotFoundError(
        f"No sentiment data found for {SCHOOL}.  Run project_part_1_data_pipeline.ipynb first."
    )

# ── Aggregate, combine, classify ──────────────────────────────────────────
# Combining both datasets before classification means every bubble across
# both maps shares the same Jenks bin boundaries for size and color.
_df_jmu    = filter_places(aggregate_places(_df_jmu_raw,    sentiment=True), MIN_COUNT, PLACE_TYPES)
_df_school = filter_places(aggregate_places(_df_school_raw, sentiment=True), MIN_COUNT, PLACE_TYPES)
_n_jmu     = len(_df_jmu)
_df_all    = pd.concat([_df_jmu, _df_school], ignore_index=True)

_labels, _cmap = jenks_sentiment_classify(_df_all, COLOR_SCALE, N_SIZE_CLASSES, N_COLOR_BUCKETS)

# Split back with shared size_class + color_class intact
df_jmu_comp    = _df_all.iloc[:_n_jmu].copy()
df_school_comp = _df_all.iloc[_n_jmu:].copy()
print(f'JMU: {len(df_jmu_comp):,} places  |  {SCHOOL}: {len(df_school_comp):,} places')
print(f'Shared sentiment bins: {_labels}')


# ── JMU map ───────────────────────────────────────────────────────────────
px.scatter_map(
    df_jmu_comp,
    lat="latitude", lon="longitude",
    size="size_class", color="color_class",
    hover_name="place",
    hover_data={
        "avg_roberta_compound": ":.3f",
        "location_count": True,
        "size_class": False,
        "color_class": False,
        "latitude": False,
        "longitude": False,
    },
    color_discrete_map=_cmap,
    category_orders={
        "color_class": _labels,
    },
    size_max=SIZE_MAX, map_style=MAP_STYLE, center=CENTER, zoom=ZOOM, height=650,
    title="JMU — Comparative Sentiment Map",  # ← change this title
).update_layout(
    margin={
        "r": 0,
        "t": 50,
        "l": 0,
        "b": 0,
    }
).show()

JMU: 101 places  |  VCU: 95 places
Shared sentiment bins: ['-0.73 to -0.34', '-0.34 to -0.11', '-0.11 to 0.08', '0.08 to 0.35', '0.35 to 0.97']


### VCU <- Change this


📋 **Instructions:**

**Task**: Write an analysis of the sentiment map of your school that tries to answer your hypothesis.
**Length**: 4-5 sentences
**Description**: After giving your overview and your highlight of three locations, you are going to analyze the sentiment map in terms of your hypothesis. The key is to make sure that the map and your analysis match. Rather than trying to tell a "global" story, you want to tell a detailed story about one specific issue. You want the map to show the minimum number of locations to tell this story. This story will be the evidence for or against your hypothesis. If the evidence contradicts your hypothesis that is a sign of good scholarship, don't be afraid to contradict your earlier assessment.


[The VCU sentiment results reveal a more mixed pattern, with negative sentiment appearing across several locations and topics. Gaza and the State of Israel had the lowest average sentiment scores at -0.602 and -0.498, respectively, showing that some conversations connected to international events contributed to negative sentiment in the dataset. Among locations more directly connected to campus, Monroe Park, Richmond, VA, had a negative score of -0.386, while the VCU University Student Commons and Gladding Residence Center also showed negative average scores. However, Monroe Park under a separate location entry had a slightly positive score of 0.069, demonstrating that sentiment can vary even when the same physical location is identified under different names. Overall, the findings partially support our hypothesis that VCU's urban environment would produce more varied geographic discussions, but they do not establish that city life itself caused more negative sentiment.]

In [13]:
# Run the cell above first — this cell reuses df_school_comp, _cmap, _labels,
# and the map settings (SIZE_MAX, MAP_STYLE, CENTER, ZOOM) already set there.
px.scatter_map(
    df_school_comp,
    lat="latitude",
    lon="longitude",
    size="size_class",
    color="color_class",
    hover_name="place",
    hover_data={
        "avg_roberta_compound": ":.3f",
        "location_count": True,
        "size_class": False,
        "color_class": False,
        "latitude": False,
        "longitude": False,
    },
    color_discrete_map=_cmap,
    category_orders={
        "color_class": _labels,
    },
    size_max=SIZE_MAX,
    map_style=MAP_STYLE,
    center=CENTER,
    zoom=ZOOM,
    height=650,
    title=f"{chart_title_prefix} — Sentiment Map",
).update_layout(
    margin={
        "r": 0,
        "t": 50,
        "l": 0,
        "b": 0,
    }
).show()

In [14]:
# Compare the strongest sentiment locations at both schools

for school_name, data in [("JMU", df_jmu_comp), ("VCU", df_school_comp)]:
    print("\n", school_name, "- MOST NEGATIVE LOCATIONS")
    display(
        data[data["location_count"] >= 5]
        .sort_values("avg_roberta_compound")
        [["place", "location_count", "avg_roberta_compound"]]
        .head(10)
    )

    print("\n", school_name, "- MOST POSITIVE LOCATIONS")
    display(
        data[data["location_count"] >= 5]
        .sort_values("avg_roberta_compound", ascending=False)
        [["place", "location_count", "avg_roberta_compound"]]
        .head(10)
    )


 JMU - MOST NEGATIVE LOCATIONS


,place,location_count,avg_roberta_compound
7,Southview Apartments,11,-0.347246
2,United States,46,-0.300805
10,New Jersey,13,-0.279451
9,Eagle Hall,8,-0.275583
3,Harrsionburg,44,-0.212182
27,"Hillside Hall, Harrisonburg, VA",5,-0.190247
12,East Campus (James Madison University),7,-0.180015
24,North Carolina,5,-0.173890
18,University of North Texas,5,-0.163477
8,D-Hall (JMU),19,-0.149545



 JMU - MOST POSITIVE LOCATIONS


,place,location_count,avg_roberta_compound
17,Bridgeforth Stadium,5,0.405977
5,"University of Oregon, Oregon",10,0.327331
32,"Bridgewater, Virginia",5,0.239731
44,Newman Lake (JMU),5,0.146243
6,District of Columbia,10,0.123157
11,West Virginia,7,0.058213
15,University of Oregon Autzen Stadium,5,0.052472
13,Richmond,7,0.022341
20,Charlottesville,5,0.021084
64,"Chandler Hall, JMU",6,-0.013453



 VCU - MOST NEGATIVE LOCATIONS


,place,location_count,avg_roberta_compound
113,Gaza,8,-0.601523
107,State of Israel,30,-0.497585
118,"Monroe Park, Richmond, VA",9,-0.386325
104,United States,45,-0.354299
105,Gaza Strip,7,-0.350543
106,Palestine,25,-0.344165
111,Charlottesville,6,-0.239284
124,VCU University Student Commons,9,-0.195637
158,VCU Gladding Residence Center,11,-0.189229
121,"Richmond, VA",15,-0.170243



 VCU - MOST POSITIVE LOCATIONS


,place,location_count,avg_roberta_compound
129,New York City,7,0.253006
155,District of Columbia,8,0.149461
162,"The James Apartments, Richmond, VA",6,0.083091
112,Monroe Park,45,0.068728
149,VCU Rhoads Hall,7,0.065393
148,"Gladding Residence Center, VCU",17,0.032830
102,"Richmond, Virginia",26,-0.054759
136,"Chesterfield County, Virginia",5,-0.059339
101,Richmond,189,-0.075158
103,Virginia,56,-0.092928


## Conclusion and Further Research

📋 **Instructions:**

**Task**: Write a conclusion that ties everything together.
**Length**: 5-8 sentences (1-2 paragraphs)
**Description**: Write a conclusion in which you lead with your initial hypothesis and explain whether the data proves, disproves or nuances your hypothesis. For example, you may have found that there is a strong focus on the campus community at JMU, but did not necessarily find that a commuter school had significantly less of a community focus. In this case, you neither proved nor disproved your thesis, but rather nuanced it a bit. You should round out your conclusion by thinking through what type of data you could have collected to help prove your point more. This is your nod towards further research.




## Conclusion and Further Research

Our initial hypothesis suggested that VCU's urban environment would produce a wider range of geographic discussions and more negative sentiment, while JMU's traditional college-town environment would produce more positive conversations centered around campus life. The findings partially support this hypothesis, as both universities showed distinct patterns in the locations and topics discussed on Reddit. JMU had particularly positive sentiment surrounding Bridgeforth Stadium, while some student housing locations, including Southview Apartments and Eagle Hall, showed more negative sentiment. VCU conversations included both positive and negative sentiment around campus locations, while discussions about international locations also contributed to the overall negative results. These findings suggest that university environments may influence online conversations, but they do not prove that an urban or college-town setting directly determines sentiment. 

Further research could examine the actual Reddit posts and comments associated with specific locations to better understand why certain places receive positive or negative sentiment. Collecting additional data from student surveys, campus reviews, and other social media platforms could also provide a more complete understanding of student experiences. Finally, improving location cleaning by combining duplicate place names would make future geographic and sentiment comparisons more accurate.